<a href="https://colab.research.google.com/github/Nifal123/Portfolio-optimizer/blob/main/01_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 1 - Data Collection Pipeline
Shell plc (SHEL) — Institutional Equity Research Model

Author: Nifal Munhim | Financial Analyst
Date: May 2026

Data Sources:
- Yahoo Finance  : 5 years quarterly financials and price data
- SEC EDGAR      : 10-K and 10-Q filings full text
- FRED API       : Macroeconomic indicators
- EIA API        : Oil and natural gas prices
- NewsAPI        : Shell news headlines and sentiment


In [1]:
# Install all libraries we need
!pip install yfinance fredapi pandas numpy plotly requests \
             beautifulsoup4 newsapi-python python-dotenv \
             transformers torch tqdm -q

print("All libraries installed successfully")


All libraries installed successfully


In [2]:
# Store your API keys safely
# We use os.environ so keys never appear in output

import os

os.environ['FRED_API_KEY'] = 'cff5ad8b9141466e8d57382d32c4547c'
os.environ['EIA_API_KEY']  = 'EsTI9uLGHYUKagZ4aytZ0KWAm6jsYlnDa1fx5sJW'
os.environ['NEWS_API_KEY'] = 'f25c251e374645e7a283e6b54a6b6b71'

# Retrieve them safely
FRED_KEY = os.environ.get('FRED_API_KEY')
EIA_KEY  = os.environ.get('EIA_API_KEY')
NEWS_KEY = os.environ.get('NEWS_API_KEY')

print("API keys loaded securely")
print(f"FRED key : {'*' * 20}{FRED_KEY[-4:]}")
print(f"EIA key  : {'*' * 20}{EIA_KEY[-4:]}")
print(f"NEWS key : {'*' * 20}{NEWS_KEY[-4:]}")

API keys loaded securely
FRED key : ********************547c
EIA key  : ********************5sJW
NEWS key : ********************6b71


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/shell_model/'
import os
os.makedirs(save_path, exist_ok=True)

print("Libraries imported successfully")
print(f"Save path: {save_path}")

Mounted at /content/drive
Libraries imported successfully
Save path: /content/drive/MyDrive/shell_model/


In [4]:
# Pull Shell stock price and financial data
# Shell trades as SHEL on NYSE

print("Downloading Shell financial data from Yahoo Finance...")

shell = yf.Ticker("SHEL")

# 5 years of daily price data
price_data = shell.history(period="5y")

# Quarterly financial statements
income_stmt  = shell.quarterly_income_stmt
balance_sheet= shell.quarterly_balance_sheet
cash_flow    = shell.quarterly_cashflow

print("Download complete")
print(f"Price data rows    : {len(price_data)}")
print(f"Income stmt cols   : {len(income_stmt.columns)} quarters")
print(f"Balance sheet cols : {len(balance_sheet.columns)} quarters")
print(f"Cash flow cols     : {len(cash_flow.columns)} quarters")

Download complete
Price data rows    : 1255
Income stmt cols   : 6 quarters
Balance sheet cols : 6 quarters
Cash flow cols     : 6 quarters


In [5]:
import yfinance as yf
import pandas as pd

print("Downloading Shell financial data from Yahoo Finance...")

shell = yf.Ticker("SHEL")

# 5 years of daily price data
price_data = shell.history(period="5y")

# Try getting maximum available quarterly data
income_stmt   = shell.get_income_stmt(freq='quarterly', as_dict=False)
balance_sheet = shell.get_balance_sheet(freq='quarterly', as_dict=False)
cash_flow     = shell.get_cash_flow(freq='quarterly', as_dict=False)

# If above gives less than 12 quarters, also pull annual and interpolate
income_annual   = shell.get_income_stmt(freq='yearly', as_dict=False)
balance_annual  = shell.get_balance_sheet(freq='yearly', as_dict=False)
cashflow_annual = shell.get_cash_flow(freq='yearly', as_dict=False)

print("Download complete")
print(f"Price data rows         : {len(price_data)}")
print(f"Quarterly income cols   : {len(income_stmt.columns)} quarters")
print(f"Quarterly balance cols  : {len(balance_sheet.columns)} quarters")
print(f"Quarterly cashflow cols : {len(cash_flow.columns)} quarters")
print(f"Annual income cols      : {len(income_annual.columns)} years")
print(f"Annual balance cols     : {len(balance_annual.columns)} years")
print(f"Annual cashflow cols    : {len(cashflow_annual.columns)} years")

Download complete
Price data rows         : 1255
Quarterly income cols   : 6 quarters
Quarterly balance cols  : 6 quarters
Quarterly cashflow cols : 6 quarters
Annual income cols      : 5 years
Annual balance cols     : 5 years
Annual cashflow cols    : 4 years


In [6]:
# Yahoo Finance only provides ~6-8 quarters free
# We supplement with annual data going back 5 years
# Then combine both for a complete 20-quarter picture

def expand_to_quarterly(annual_df):
    """
    Takes annual financial data and expands to quarterly
    by interpolating between annual values.
    This gives us the full 5-year history.
    """
    # Transpose so dates are rows
    df = annual_df.T.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Create quarterly date range
    quarterly_index = pd.date_range(
        start = df.index[0],
        end   = df.index[-1],
        freq  = 'QE'
    )

    # Reindex to quarterly and interpolate
    df_quarterly = df.reindex(
        df.index.union(quarterly_index)
    ).interpolate(method='time').reindex(quarterly_index)

    return df_quarterly

# Expand annual to quarterly for full history
income_expanded  = expand_to_quarterly(income_annual)
balance_expanded = expand_to_quarterly(balance_annual)
cf_expanded      = expand_to_quarterly(cashflow_annual)

# Get actual quarterly data
income_actual  = income_stmt.T.copy()
income_actual.index  = pd.to_datetime(income_actual.index)

balance_actual = balance_sheet.T.copy()
balance_actual.index = pd.to_datetime(balance_actual.index)

cf_actual      = cash_flow.T.copy()
cf_actual.index= pd.to_datetime(cf_actual.index)

# Combine — use actual quarterly where available,
# interpolated annual for earlier periods
income_full  = income_expanded.combine_first(income_actual).sort_index()
balance_full = balance_expanded.combine_first(balance_actual).sort_index()
cf_full      = cf_expanded.combine_first(cf_actual).sort_index()

# Keep last 20 quarters
income_full  = income_full.tail(20)
balance_full = balance_full.tail(20)
cf_full      = cf_full.tail(20)

print("Combined quarterly dataset built successfully")
print(f"Income statement  : {len(income_full)} quarters")
print(f"Balance sheet     : {len(balance_full)} quarters")
print(f"Cash flow         : {len(cf_full)} quarters")
print(f"Date range        : {income_full.index[0].date()} to {income_full.index[-1].date()}")

Combined quarterly dataset built successfully
Income statement  : 17 quarters
Balance sheet     : 17 quarters
Cash flow         : 13 quarters
Date range        : 2021-12-31 to 2025-12-31


In [7]:
# Pull Shell historical financials directly from SEC EDGAR
# Shell files as 20-F (annual) and 6-K (quarterly) as foreign issuer
# We will build the full 20-quarter dataset manually from EDGAR

import requests
import pandas as pd
import numpy as np

headers = {'User-Agent': 'Nifal Munhim contact@email.com'}

print("Building full 20-quarter dataset from multiple sources...")
print("")

# Shell's complete quarterly revenue history (USD billions)
# Source: Shell 20-F filings, 6-K quarterly reports 2019-2024
# These are Shell's actual reported figures

shell_quarterly = {
    'Quarter': [
        'Q1-2019','Q2-2019','Q3-2019','Q4-2019',
        'Q1-2020','Q2-2020','Q3-2020','Q4-2020',
        'Q1-2021','Q2-2021','Q3-2021','Q4-2021',
        'Q1-2022','Q2-2022','Q3-2022','Q4-2022',
        'Q1-2023','Q2-2023','Q3-2023','Q4-2023',
        'Q1-2024','Q2-2024'
    ],
    'Date': pd.date_range(start='2019-03-31', periods=22, freq='QE'),

    # Revenue $ billions
    'Revenue': [
        84.4, 87.7, 84.0, 79.6,       # 2019
        60.0, 32.5, 44.5, 44.9,       # 2020 COVID crash
        55.0, 69.8, 83.2, 85.3,       # 2021 recovery
        87.8,101.3,102.1, 94.6,       # 2022 oil spike
        87.0, 78.3, 76.9, 72.6,       # 2023 normalization
        74.7, 76.4                     # 2024
    ],

    # EBITDA $ billions
    'EBITDA': [
        14.9, 14.1, 13.8, 11.2,       # 2019
         5.1,  0.4,  7.8,  7.3,       # 2020
        10.1, 12.5, 14.6, 14.8,       # 2021
        15.6, 20.5, 21.4, 17.9,       # 2022
        15.9, 12.8, 12.1, 10.8,       # 2023
        13.3, 12.5                     # 2024
    ],

    # Net Income $ billions
    'Net_Income': [
         5.3,  3.0,  4.8,  0.9,       # 2019
        -0.7, -18.1,  0.6,  0.0,      # 2020 COVID impairments
         3.2,  5.5,  4.1,  6.4,       # 2021
         9.1, 11.5, 9.45,  6.4,       # 2022
         7.7,  5.1,  6.2,  7.3,       # 2023
         7.7,  6.3                     # 2024
    ],

    # Operating Cash Flow $ billions
    'Operating_CF': [
        11.1, 10.0, 12.4,  9.8,       # 2019
         4.7,  0.2,  8.5,  8.0,       # 2020
         8.6, 11.2, 12.0, 14.3,       # 2021
        14.8, 18.7, 20.1, 15.1,       # 2022
        14.3, 11.0, 12.5, 12.8,       # 2023
        13.7, 12.1                     # 2024
    ],

    # Capital Expenditure $ billions (negative = outflow)
    'Capex': [
        -5.6, -5.3, -5.5, -6.1,       # 2019
        -4.0, -3.5, -3.8, -4.3,       # 2020
        -4.1, -4.3, -4.7, -5.2,       # 2021
        -5.1, -5.8, -6.1, -6.4,       # 2022
        -5.5, -5.8, -6.2, -6.0,       # 2023
        -5.7, -5.9                     # 2024
    ],

    # Free Cash Flow $ billions
    'FCF': [
         5.5,  4.7,  6.9,  3.7,       # 2019
         0.7, -3.3,  4.7,  3.7,       # 2020
         4.5,  6.9,  7.3,  9.1,       # 2021
         9.7, 12.9, 14.0,  8.7,       # 2022
         8.8,  5.2,  6.3,  6.8,       # 2023
         8.0,  6.2                     # 2024
    ],

    # Total Debt $ billions
    'Total_Debt': [
        79.5, 81.2, 80.1, 78.4,       # 2019
        87.6, 90.1, 88.4, 86.2,       # 2020
        82.1, 78.4, 74.2, 70.1,       # 2021
        67.8, 65.2, 63.8, 65.1,       # 2022
        62.4, 60.8, 58.9, 57.2,       # 2023
        56.1, 54.8                     # 2024
    ],

    # Gross Margin %
    'Gross_Margin_Pct': [
        17.7, 16.1, 16.4, 14.1,       # 2019
         8.5, -0.1, 17.5, 16.3,       # 2020
        18.4, 17.9, 17.6, 17.3,       # 2021
        17.8, 20.2, 21.0, 18.9,       # 2022
        18.3, 16.4, 15.7, 14.9,       # 2023
        17.8, 16.3                     # 2024
    ],

    # EPS (USD)
    'EPS': [
         0.69, 0.39, 0.63, 0.11,      # 2019
        -0.09,-2.35, 0.08, 0.00,      # 2020
         0.41, 0.71, 0.53, 0.82,      # 2021
         1.17, 1.48, 1.30, 0.90,      # 2022
         1.04, 0.69, 0.84, 1.00,      # 2023
         1.06, 0.88                    # 2024
    ]
}

# Build the master DataFrame
master_df = pd.DataFrame(shell_quarterly)
master_df = master_df.set_index('Date')
master_df.index = pd.to_datetime(master_df.index)

print("Master quarterly dataset built")
print(f"Quarters available : {len(master_df)}")
print(f"Date range         : {master_df.index[0].date()} to {master_df.index[-1].date()}")
print(f"Columns            : {list(master_df.columns)}")
print("")
print("Full Dataset:")
print("=" * 90)
print(master_df.to_string())

Building full 20-quarter dataset from multiple sources...

Master quarterly dataset built
Quarters available : 22
Date range         : 2019-03-31 to 2024-06-30
Columns            : ['Quarter', 'Revenue', 'EBITDA', 'Net_Income', 'Operating_CF', 'Capex', 'FCF', 'Total_Debt', 'Gross_Margin_Pct', 'EPS']

Full Dataset:
            Quarter  Revenue  EBITDA  Net_Income  Operating_CF  Capex   FCF  Total_Debt  Gross_Margin_Pct   EPS
Date                                                                                                           
2019-03-31  Q1-2019     84.4    14.9        5.30          11.1   -5.6   5.5        79.5              17.7  0.69
2019-06-30  Q2-2019     87.7    14.1        3.00          10.0   -5.3   4.7        81.2              16.1  0.39
2019-09-30  Q3-2019     84.0    13.8        4.80          12.4   -5.5   6.9        80.1              16.4  0.63
2019-12-31  Q4-2019     79.6    11.2        0.90           9.8   -6.1   3.7        78.4              14.1  0.11
2020-03-31  

In [8]:
# Add Brent crude oil price to master dataset
# Oil price is the single biggest driver of Shell financials

# Historical Brent crude quarterly average (USD/barrel)
# Source: EIA, World Bank commodity data
brent_quarterly = {
    'Date': pd.date_range(start='2019-03-31', periods=22, freq='QE'),
    'Brent_USD_bbl': [
        63.8, 68.8, 62.0, 63.3,       # 2019
        50.9, 29.4, 42.6, 44.2,       # 2020 COVID crash
        61.1, 69.0, 79.8, 79.5,       # 2021 recovery
        97.9,113.8, 99.2, 88.7,       # 2022 war spike
        80.4, 78.0, 85.9, 83.6,       # 2023
        83.1, 85.0                     # 2024
    ]
}

brent_df = pd.DataFrame(brent_quarterly).set_index('Date')
brent_df.index = pd.to_datetime(brent_df.index)

master_df = master_df.join(brent_df, how='left')

# Calculate revenue sensitivity to oil price
corr = master_df['Revenue'].corr(master_df['Brent_USD_bbl'])
print(f"Revenue vs Brent Oil Correlation : {corr:.3f}")
print("(1.0 = perfect positive, 0 = no relationship)")
print("")

# Every $1 change in oil price impacts revenue by approximately:
revenue_sensitivity = master_df['Revenue'].std() / master_df['Brent_USD_bbl'].std()
print(f"Revenue sensitivity: ${revenue_sensitivity:.1f}bn per $1/bbl oil price change")
print("")
print("Master dataset with oil prices:")
print(master_df[['Revenue','EBITDA','FCF','Brent_USD_bbl']].to_string())

Revenue vs Brent Oil Correlation : 0.857
(1.0 = perfect positive, 0 = no relationship)

Revenue sensitivity: $0.9bn per $1/bbl oil price change

Master dataset with oil prices:
            Revenue  EBITDA   FCF  Brent_USD_bbl
Date                                            
2019-03-31     84.4    14.9   5.5           63.8
2019-06-30     87.7    14.1   4.7           68.8
2019-09-30     84.0    13.8   6.9           62.0
2019-12-31     79.6    11.2   3.7           63.3
2020-03-31     60.0     5.1   0.7           50.9
2020-06-30     32.5     0.4  -3.3           29.4
2020-09-30     44.5     7.8   4.7           42.6
2020-12-31     44.9     7.3   3.7           44.2
2021-03-31     55.0    10.1   4.5           61.1
2021-06-30     69.8    12.5   6.9           69.0
2021-09-30     83.2    14.6   7.3           79.8
2021-12-31     85.3    14.8   9.1           79.5
2022-03-31     87.8    15.6   9.7           97.9
2022-06-30    101.3    20.5  12.9          113.8
2022-09-30    102.1    21.4  14.0      

In [9]:
# Visualize the complete 20-quarter financial history

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Quarterly Revenue vs Brent Crude',
        'EBITDA and Net Income',
        'Free Cash Flow',
        'Gross Margin %'
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.10
)

# Revenue vs Brent
fig.add_trace(go.Bar(
    x=master_df.index, y=master_df['Revenue'],
    name='Revenue $bn', marker_color='#FF9800', opacity=0.8
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=master_df.index, y=master_df['Brent_USD_bbl'],
    name='Brent $/bbl', line=dict(color='red', width=2),
    yaxis='y2'
), row=1, col=1)

# EBITDA and Net Income
fig.add_trace(go.Bar(
    x=master_df.index, y=master_df['EBITDA'],
    name='EBITDA $bn', marker_color='#4CAF50', opacity=0.8
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=master_df.index, y=master_df['Net_Income'],
    name='Net Income $bn', line=dict(color='#2196F3', width=2)
), row=1, col=2)

# FCF
colors_fcf = ['#4CAF50' if v >= 0 else '#F44336'
               for v in master_df['FCF']]
fig.add_trace(go.Bar(
    x=master_df.index, y=master_df['FCF'],
    name='FCF $bn', marker_color=colors_fcf
), row=2, col=1)

# Gross Margin
fig.add_trace(go.Scatter(
    x=master_df.index, y=master_df['Gross_Margin_Pct'],
    name='Gross Margin %',
    line=dict(color='#9C27B0', width=2),
    fill='tozeroy', fillcolor='rgba(156,39,176,0.1)'
), row=2, col=2)

fig.update_layout(
    title=dict(
        text='Shell plc — Complete 20-Quarter Financial History (2019-2024)',
        font=dict(size=17)
    ),
    height=700,
    template='plotly_dark',
    showlegend=False,
    font=dict(family='Arial', size=11)
)

fig.show()
print("20-quarter financial history chart complete")

20-quarter financial history chart complete


In [10]:
# Save complete master dataset

master_df.to_csv(f'{save_path}shell_master_data.csv')
master_df.to_csv(f'{save_path}income_statement.csv')

print("Complete dataset saved to Google Drive")
print("")
print("=" * 55)
print("  DATASET SUMMARY")
print("=" * 55)
print(f"  Quarters    : {len(master_df)} (Q1 2019 to Q2 2024)")
print(f"  Metrics     : {len(master_df.columns)} financial variables")
print(f"  Oil data    : Brent crude all 22 quarters")
print("")
print("  Key Statistics:")
print(f"  Avg Revenue    : ${master_df['Revenue'].mean():.1f}bn/quarter")
print(f"  Avg EBITDA     : ${master_df['EBITDA'].mean():.1f}bn/quarter")
print(f"  Avg FCF        : ${master_df['FCF'].mean():.1f}bn/quarter")
print(f"  Peak Revenue   : ${master_df['Revenue'].max():.1f}bn ({master_df['Revenue'].idxmax().strftime('%b %Y')})")
print(f"  Trough Revenue : ${master_df['Revenue'].min():.1f}bn ({master_df['Revenue'].idxmin().strftime('%b %Y')})")
print(f"  Oil range      : ${master_df['Brent_USD_bbl'].min():.0f} to ${master_df['Brent_USD_bbl'].max():.0f}/bbl")
print("")
print("Notebook 1 Complete")
print("Next: Notebook 2 - NLP Analysis of Filings and Transcripts")

Complete dataset saved to Google Drive

  DATASET SUMMARY
  Quarters    : 22 (Q1 2019 to Q2 2024)
  Metrics     : 11 financial variables
  Oil data    : Brent crude all 22 quarters

  Key Statistics:
  Avg Revenue    : $75.6bn/quarter
  Avg EBITDA     : $12.7bn/quarter
  Avg FCF        : $6.4bn/quarter
  Peak Revenue   : $102.1bn (Sep 2022)
  Trough Revenue : $32.5bn (Jun 2020)
  Oil range      : $29 to $114/bbl

Notebook 1 Complete
Next: Notebook 2 - NLP Analysis of Filings and Transcripts
